# Prompt 1 completion — explicit quick, smoke, and full workflows

This notebook does not retrain Track A, modify nnU-Net checkpoints, or open the 52-case S cohort. Quick and smoke actions never build the 480-case manifest. All persistent state and artifacts stay on Drive.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

# Single editable configuration cell.
DRIVE_ROOT = Path('/content/drive/MyDrive/ToothFairy/ToothFairy3/iac_runs')
DATASET_ROOT = DRIVE_ROOT / 'dataset_cache_colab_v1/Dataset801_IAC_LR'
NNUNET_RESULTS = DRIVE_ROOT / 'nnUNet_results'
TRACKB_CACHE_ROOT = DRIVE_ROOT / 'sdf_cache_backup'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
SPLITS_PATH = DRIVE_ROOT / 'configs_cache/splits.json'
REPO_URL = 'https://github.com/ColdVI/ToothFairy3-IAC-Segmentation-Flow.git'
PINNED_COMMIT = 'REPLACE_WITH_PROMPT1_COMMIT_SHA'
NUM_WORKERS = 2
DEVICE = 'cuda'
QUICK_PREFLIGHT_CASES_PER_FOLD = 1
FULL_PREFLIGHT_CASES = 40
PREFLIGHT_MODE = 'quick'
MAX_CASES = 2  # retained for config compatibility; smoke is explicitly two cases
FORCE_REBUILD = False
EXPORT_TRUE_SOFTMAX = True
RETRY_FAILED = True

REPO = Path('/content/ToothFairy3-IAC-Segmentation-Flow')
CONFIG_PATH = Path('/content/prompt1_completion_config.json')

def bootstrap_repo():
    if PINNED_COMMIT.startswith('REPLACE_'):
        raise ValueError('Set PINNED_COMMIT to the delivered Prompt-1 commit SHA')
    if not REPO.is_dir():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
    subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO, check=True)
    head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    wanted = subprocess.check_output(['git', 'rev-parse', PINNED_COMMIT], cwd=REPO, text=True).strip()
    assert head == wanted, (head, wanted)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO/'requirements.txt')], check=True)
    return REPO

def write_runner_config():
    values = {
        'DRIVE_ROOT': DRIVE_ROOT, 'DATASET_ROOT': DATASET_ROOT,
        'NNUNET_RESULTS': NNUNET_RESULTS, 'TRACKB_CACHE_ROOT': TRACKB_CACHE_ROOT,
        'OUTPUT_ROOT': OUTPUT_ROOT, 'SPLITS_PATH': SPLITS_PATH,
        'PINNED_COMMIT': PINNED_COMMIT, 'NUM_WORKERS': NUM_WORKERS,
        'DEVICE': DEVICE, 'QUICK_PREFLIGHT_CASES_PER_FOLD': QUICK_PREFLIGHT_CASES_PER_FOLD,
        'FULL_PREFLIGHT_CASES': FULL_PREFLIGHT_CASES, 'PREFLIGHT_MODE': PREFLIGHT_MODE,
        'MAX_CASES': MAX_CASES, 'FORCE_REBUILD': FORCE_REBUILD,
        'EXPORT_TRUE_SOFTMAX': EXPORT_TRUE_SOFTMAX, 'RETRY_FAILED': RETRY_FAILED}
    payload = {name: str(value) if isinstance(value, Path) else value for name, value in values.items()}
    CONFIG_PATH.write_text(json.dumps(payload, indent=2))
    return CONFIG_PATH


## A. Setup + pytest

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = bootstrap_repo()
import torch
assert DEVICE == 'cuda' and torch.cuda.is_available(), 'A CUDA runtime is required for OOF inference'
for name, path in [('DATASET_ROOT', DATASET_ROOT), ('NNUNET_RESULTS', NNUNET_RESULTS), ('SPLITS_PATH', SPLITS_PATH)]:
    assert path.exists(), f'{name} missing: {path}'
for path in (TRACKB_CACHE_ROOT, OUTPUT_ROOT):
    path.mkdir(parents=True, exist_ok=True)
    assert path.resolve().is_relative_to(DRIVE_ROOT.resolve())
cfg = write_runner_config()
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests'], cwd=REPO, check=True)


## B. QUICK PREFLIGHT
One deterministic legacy-complete validation case per fold; no 480-case manifest.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'preflight-quick'], cwd=REPO, check=True)


## C. TWO-CASE SMOKE
Runs quick preflight, then exactly two development cases with resumable OOF/SDF validation.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'smoke'], cwd=REPO, check=True)


## D. FULL PREFLIGHT
Runs the sealed 40-case scientific provenance, audit, and three-path identity gate.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'preflight-full'], cwd=REPO, check=True)


## E. FULL COMPLETION
Builds the 480-case manifest, completes missing OOF/SDF artifacts, rebuilds the manifest, and runs full-CV identity. Safe to resume after disconnects.

In [ ]:
cfg = write_runner_config()
subprocess.run([sys.executable, 'scripts/prompt1_completion.py', '--config', str(cfg), 'run-full'], cwd=REPO, check=True)
# Outputs include prompt1/cache_manifest_480.json and baselines/identity_prior.json.
